# 07 - M7: concept readout + 24x24 map (real CLIP, real T4)
**Runtime > T4 GPU.** Gate: map sum plus biases equals the logit (to fp16 tolerance).
Also checked on the real pipeline: the identity holds for the real features and a live-gate steered stack, in fp32 and fp16 autocast, per concept as well as overall; the loss reaches the readout, the gates, the front end and LoRA; memory/speed with the readout on (budget: peak < 10 GiB at batch 8).

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import json, time, os, numpy as np, torch, torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPVisionModel, CLIPTextModelWithProjection, CLIPTokenizer
from src.data import load_manifest, ManifestDataset
from src.backbone import ClipLoRABackbone
from src import concepts as K
from src.fusion import ConceptFrontEnd
from src.steering import SteeredMamba
from src.readout import ConceptReadout, decomposition_error, readout_diagnostics
dev = 'cuda'
res = {'milestone': 'M7'}

ASSET = 'assets/concepts_text_768.npy'
if os.path.exists(ASSET):
    T = K.load_or_encode(ASSET)
else:
    tm = CLIPTextModelWithProjection.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32)
    T = K.load_or_encode(ASSET, tm, CLIPTokenizer.from_pretrained(K.TEXT_CHECKPOINT), device='cuda'); del tm
mf = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=mf['revision'])
tr = ManifestDataset(mf, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)
batch = [tr[i] for i in range(8)]
xb = torch.stack([b[0] for b in batch]).to(dev)
yb = torch.tensor([float(b[1]) for b in batch], device=dev)
print('labels in this batch (1 = fake):', yb.tolist())
clip = CLIPVisionModel.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32).to(dev).eval()
bb = ClipLoRABackbone(clip, taps=(16, 24), r=8, alpha=16.0).to(dev)
fe = ConceptFrontEnd(T, d_in=1024, d=384, n_taps=2).to(dev)

torch.manual_seed(0)
mamba = SteeredMamba(2, chunk=8).to(dev).eval()
torch.manual_seed(0)
head = ConceptReadout(384).to(dev)
assert sum(p.numel() for p in head.parameters()) == 6_160
print('readout params', sum(p.numel() for p in head.parameters()), '| steered stack params', sum(p.numel() for p in mamba.parameters()))

def states(autocast):
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=autocast):
        o = fe(bb(xb)); H = mamba(o['X'], o['alpha'])
    return H, o['alpha']

In [ ]:
# GATE: map sum + biases == logit, on the real pipeline. Run with the init heads AND with a randomised, biased head
# (so the '+ biases' part is exercised), with gates at init AND live, in fp32 and fp16 autocast.
# The logit is pool-then-score, the map is score-then-pool: two routes, so agreement is a real check.
FP16_TOL = 1e-3   # relative to max(1, |logit|); the readout is fp32 internally, so the observed error should be ~1e-6
g1 = {}
for gates in ('init', 'live'):
    if gates == 'live':
        with torch.no_grad():
            for p in mamba.gate_parameters(): p.normal_(0, 0.1)
    for mode in ('fp32', 'fp16'):
        H, alpha = states(mode == 'fp16')
        for head_kind in ('init', 'random_biased'):
            h_ = ConceptReadout(384).to(dev)
            if head_kind == 'random_biased':
                with torch.no_grad(): h_.w.normal_(0, 0.5); h_.b.normal_(0, 0.5)
            with torch.autocast('cuda', dtype=torch.float16, enabled=(mode == 'fp16')):
                out = h_(H, alpha)
            e = decomposition_error(h_, out)
            key = f'gates_{gates}/{mode}/head_{head_kind}'
            g1[key] = e
            print(key, {k: f'{v:.2e}' for k, v in e.items()})
            assert e['map_plus_bias_vs_logit'] <= FP16_TOL * max(1.0, e['logit_absmax']), key
            assert e['per_concept'] <= FP16_TOL * max(1.0, e['logit_absmax']), key
            assert out['map'].shape == (8, 24, 24) and out['concept_maps'].shape == (8, 16, 24, 24)
            assert torch.isfinite(out['map']).all()
res['gate_map_plus_bias_equals_logit'] = g1
print('GATE OK  | worst map+bias vs logit:', f"{max(v['map_plus_bias_vs_logit'] for v in g1.values()):.2e}")

In [ ]:
# What alpha does to the pooling on real data (descriptive; the model is untrained, so no meaning is claimed)
H, alpha = states(True)
out = head(H, alpha)
diag = readout_diagnostics(out, polarity=K.CONCEPT_POLARITY)
print(json.dumps({k: v for k, v in diag.items() if k != 'score_absmean_per_concept'}, indent=1))
print('pooling entropy / max possible:', f"{diag['pooling_entropy_mean']:.3f} / {diag['pooling_entropy_max_possible']:.3f}",
      '(max = every concept pools all 576 tokens equally -> the 16 heads see the same vector; D009 risk)')
spread = (out['h'] - out['h'].mean(1, keepdim=True)).norm(dim=-1).mean().item() / out['h'].norm(dim=-1).mean().item()
print('relative spread of the 16 pooled vectors around their mean:', f'{spread:.4f}')
res['readout_init_diagnostics'] = {**{k: v for k, v in diag.items()}, 'pooled_vector_relative_spread': spread}
print('map at init: abs max', out['map'].abs().max().item(), '| logit', [round(v, 4) for v in out['logit'].tolist()])

In [ ]:
# GRADIENTS + MEMORY, full pipeline (CLIP+LoRA -> front end -> 2 steered Mamba -> readout), B=8, fp16 autocast, fwd+bwd,
# real BCE loss on the real labels. The gates are live (nudged above) so the loss reaches the front end through steering too.
g = lambda p: p.grad is not None and p.grad.abs().sum().item() > 0
with torch.no_grad():
    for n, p in bb.named_parameters():
        if 'lora_B' in n: p.add_(0.01 * torch.randn_like(p))      # B=0 at init zeroes A's gradient
mamba.train(); head.train()
for mod in (bb, fe, mamba, head): mod.zero_grad(set_to_none=True)
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
ts = []
LOSS_SCALE = 1024.0   # what GradScaler does in M8
for _ in range(3):
    for mod in (bb, fe, mamba, head): mod.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast('cuda', dtype=torch.float16):
        o = fe(bb(xb)); H = mamba(o['X'], o['alpha']); out = head(H, o['alpha'])
    loss = F.binary_cross_entropy_with_logits(out['logit'], yb)
    (loss * LOSS_SCALE).backward()
    torch.cuda.synchronize(); ts.append(time.time() - t0)
assert torch.isfinite(out['logit']).all() and torch.isfinite(loss)
sec = float(np.median(ts)); peak = torch.cuda.max_memory_allocated() / 2**30
dead = {name: [n for n, p in mod.named_parameters() if not g(p) and not (name == 'fe' and n == 'affinity.text_embeds')]
        for name, mod in (('head', head), ('mamba', mamba), ('fe', fe))}
lora_dead = [n for n, p in bb.named_parameters() if 'lora_' in n and not g(p)]
nonfinite = [n for mod in (bb, fe, mamba, head) for n, p in mod.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]
print('loss', loss.item(), '| peak GiB', f'{peak:.2f}', '| sec/step', f'{sec:.2f}', '| img/s', f'{8 / sec:.1f}')
print('params with no gradient:', dead, '| LoRA zero-grad:', len(lora_dead), '| non-finite grads:', nonfinite)
assert not any(dead.values()) and not lora_dead and not nonfinite
assert peak < 10.0, 'over the D008 budget: use checkpoint=True'
res['grad_and_memory_b8'] = {'loss': loss.item(), 'peak_gib': peak, 'sec_per_step_b8': sec, 'img_per_s': 8 / sec,
                             'est_min_per_epoch_3240_imgs': 3240 / (8 / sec) / 60, 'dead_params': dead, 'lora_zero_grad': len(lora_dead)}
print('GRADIENTS + MEMORY OK')

In [ ]:
res['gate'] = 'PASS'
for p in (f'{DRIVE_ROOT}/runs/m7_gate.json', '/content/csm-clip/runs/m7_gate.json'):
    json.dump(res, open(p, 'w'), indent=1)
print(json.dumps(res, indent=1))
# Then: commit runs/m7_gate.json, set M7 to Done in docs/contract.md, tag m7-done.